In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, ArrayType
from pyspark.sql.functions import concat_ws, col
from sentence_transformers import SentenceTransformer, util
import numpy as np

In [9]:
# --- LOAD DATA (PySpark) ---
spark = SparkSession.builder.appName('SemanticSimilarityNLP').getOrCreate()
ingredients_schema = StructType([
    StructField('fdc_id', LongType(), False),
    StructField('description', StringType(), True),
    StructField('all_ingredients', ArrayType(StringType()), True)
 ])
ingredients_path = '../output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet/part-00000-33e03f78-d7e8-41fa-a09f-4a8c4605dc35-c000.snappy.parquet'
df_ing = spark.read.schema(ingredients_schema).parquet(ingredients_path)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/13 19:47:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/13 19:47:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [10]:
# --- CREATE TEXT FOR EMBEDDING (PySpark) ---
df_ing = df_ing.withColumn('text_for_embedding', concat_ws('. Ingredients: ', col('description'), concat_ws(', ', col('all_ingredients'))))

In [11]:
# --- LOAD EMBEDDING MODEL ---
model = SentenceTransformer('all-MiniLM-L6-v2')

In [12]:
# --- COMPUTE EMBEDDINGS (convert Spark to Pandas for embedding) ---
foods_pd = df_ing.select('fdc_id', 'description', 'all_ingredients', 'text_for_embedding').toPandas()
embeddings = model.encode(foods_pd['text_for_embedding'].tolist(), show_progress_bar=True, convert_to_numpy=True)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [13]:
# --- FUNCTION: RECOMMEND SIMILAR PRODUCTS ---
def recommend_similar_products(query_text, top_n=5):
    query_emb = model.encode([query_text], convert_to_numpy=True)[0]
    scores = util.cos_sim(query_emb, embeddings)[0].cpu().numpy() if hasattr(util.cos_sim(query_emb, embeddings)[0], 'cpu') else util.cos_sim(query_emb, embeddings)[0].numpy()
    top_idx = np.argsort(scores)[::-1][:top_n]
    return foods_pd.iloc[top_idx][['fdc_id', 'description', 'all_ingredients']], scores[top_idx]

In [15]:
# Example: Recommend products similar to a given product (Spark pipeline version)
#'PEPPERIDGE FARM BREAD GARLIC. Ingredients: enriched wheat flour, water, margarine, mozzarella cheese, yeast, sugar, garlic powder, parsley flakes'
#'APPLE, BLUEBERRY & POMEGRANATE ORGANIC BLENDED FRUIT SNACK, APPLE, BLUEBERRY & POMEGRANATE. Ingredients: organic apple puree, organic blueberry puree, organic pomegranate juice concentrate, ascorbic acid (vitamin c), organic lemon juice concentrate'
query = 'APPLE, BLUEBERRY & POMEGRANATE ORGANIC BLENDED FRUIT SNACK, APPLE, BLUEBERRY & POMEGRANATE. Ingredients: organic apple puree, organic blueberry puree, organic pomegranate juice concentrate, ascorbic acid (vitamin c), organic lemon juice concentrate'
results['similarity'] = sim_scores
display(results)

,fdc_id,description,all_ingredients,similarity
0,1105918,PEPPERIDGE FARM BREAD GARLIC,"[made from: enriched wheat flour (flour, niac...",0.917576
145,356055,PEPPERIDGE FARM BREAD FIVE CHEESE GARLIC,"[made from: enriched wheat flour (flour, niac...",0.842957
4395,2491582,"Pepperidge Farm Goldfish Cheddar Crackers, Bak...","[made with smiles and whole wheat flour, enri...",0.763401
1415,1169633,PEPPERIDGE FARM COOKIES,[made from: unbleached enriched wheat flour (f...,0.759273
3803,2231443,Pepperidge Farm Milano Cinnamon Chocolate Cook...,"[made from: enriched wheat flour (flour, niac...",0.754622
4864,2688641,Pepperidge Farm Nantucket Crispy Double Dark C...,"[made from: semi sweet chocolate (sugar, choc...",0.741796
3759,2197049,"Pepperidge Farm Goldfish Cheddar Crackers, 6.6...",[made with smiles and enriched wheat flour (fl...,0.726304
4889,2697750,"Pepperidge Farm Butter Hot Dog Buns, Top Slice...","[made from: enriched wheat flour (flour, niac...",0.723265
4397,2492601,Pepperidge Farm Goldfish Whole Grain Snack Cra...,"[made with smiles and whole wheat flour, enri...",0.720893
4765,2650506,Pepperidge Farm Captiva Dark Chocolate Cookies...,"[made from: enriched wheat flour (flour, niac...",0.705565
